# Eksplorasi Data (EDA) — PRDECT-ID

Notebook ini mengeksplorasi dataset **PRDECT-ID** (ulasan produk Tokopedia) sebelum pemodelan:
distribusi label sentimen & emosi, panjang teks, dan contoh data. Temuan di sini menjadi dasar
keputusan pra-pemrosesan & evaluasi (lihat README).

Jalankan dari root proyek: `jupyter notebook notebooks/01_eksplorasi_data.ipynb`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('..') / 'data' / 'prdect_id.csv'
df = pd.read_csv(DATA)
print('Bentuk data:', df.shape)
df.head()

In [ ]:
# Kolom & nilai kosong
print('Kolom:', list(df.columns))
print('\nJumlah nilai kosong per kolom penting:')
print(df[['Customer Review', 'Sentiment', 'Emotion']].isna().sum())

## Distribusi label Sentimen

In [ ]:
sent = df['Sentiment'].value_counts()
print(sent)
sent.plot(kind='bar', color=['#d62728', '#2ca02c'], title='Distribusi Sentimen')
plt.ylabel('Jumlah ulasan'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

Sentimen relatif **seimbang** (Negative vs Positive). Karena itu *accuracy* cukup informatif
untuk tugas sentimen.

## Distribusi label Emosi

In [ ]:
emo = df['Emotion'].value_counts()
print(emo)
emo.plot(kind='bar', color='#1f77b4', title='Distribusi Emosi')
plt.ylabel('Jumlah ulasan'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

Emosi **tidak seimbang** (Happy dominan, Anger paling sedikit). Implikasinya: gunakan
**macro-F1** (bukan hanya accuracy) dan `class_weight='balanced'` saat melatih model emosi.

## Panjang teks ulasan

In [ ]:
lengths = df['Customer Review'].astype(str).str.split().apply(len)
print(lengths.describe())
lengths.plot(kind='hist', bins=40, title='Distribusi panjang ulasan (jumlah kata)')
plt.xlabel('Jumlah kata'); plt.tight_layout(); plt.show()

## Contoh ulasan per kelas

In [ ]:
for emosi in df['Emotion'].unique():
    contoh = df[df['Emotion'] == emosi]['Customer Review'].iloc[0]
    print(f'[{emosi}] {contoh[:120]}')
    print('-' * 70)

## Ringkasan temuan

1. **5.400 ulasan**, tidak ada nilai kosong pada kolom teks/label.
2. **Sentimen seimbang** -> accuracy memadai.
3. **Emosi tidak seimbang** -> pakai macro-F1 + `class_weight='balanced'`.
4. Ulasan umumnya **pendek** -> TF-IDF dengan unigram+bigram cocok.
5. Langkah pra-pemrosesan (lihat `src/preprocessing.py`): case folding, hapus URL/angka/simbol,
   hapus stopword (kecuali kata negasi), stemming Sastrawi.